# OR Intelligence · 00 · Download CholecInstanceSeg (images + annotations)

Downloads the CholecInstanceSeg dataset from Synapse **inside Colab** (Google's network, not yours):

| File | Synapse ID | Size |
|---|---|---|
| Reference images `cholecinstance_seg_reference_image_set.zip` | syn70984058 | 22.3 GB |
| Annotations `cholecinstanceseg.zip` | syn66477541 | 71 MB |
| Metadata `cholecinstanceseg_metadata.csv` | syn66477699 | 5 MB |

Then it unzips both and **copies** each video's `ann_dir` next to its images, giving
`<split>/<video>/ann_dir/*.json` beside the frames.

**Before you run:** you need a Synapse Personal Access Token with *Download* permission
(synapse.org → Account Settings → Personal Access Tokens). You'll paste it into a hidden prompt; it is not saved in the notebook.

Runtime → Run all. Expect ~10–30 min for the download depending on Colab's connection.

## 1 · Settings

In [ ]:
WORK = "/content/cholecinstanceseg"                     # fast local Colab disk
DRIVE_DIR = "/content/drive/MyDrive/or-intelligence"    # same team folder as 01_Data
SAVE_IMAGE_ZIP_TO_DRIVE = True    # copies the 22.3 GB zip to Drive (needs that much free Drive space)
SAVE_ANNOTATIONS_TO_DRIVE = True  # small: annotation zip + metadata CSV

FILES = {
    "images":      "syn70984058",
    "annotations": "syn66477541",
    "metadata":    "syn66477699",
}

## 2 · Install the Synapse client and log in

In [ ]:
!pip -q install synapseclient

import getpass, synapseclient
syn = synapseclient.Synapse(skip_checks=True)
syn.login(authToken=getpass.getpass("Synapse Personal Access Token: "))

## 3 · Check free disk space

The zip (22.3 GB) plus its extracted contents need roughly 50 GB.

In [ ]:
import shutil, os
os.makedirs(WORK, exist_ok=True)
free_gb = shutil.disk_usage(WORK).free / 1e9
print(f"Free space on Colab disk: {free_gb:.0f} GB")
assert free_gb > 50, "Not enough space - switch to a runtime with a bigger disk, or set it to delete the zip after unzipping."

## 4 · Download (skips files that are already there)

In [ ]:
import time
DL = f"{WORK}/downloads"
paths = {}
for key, syn_id in FILES.items():
    t = time.time()
    ent = syn.get(syn_id, downloadLocation=DL, ifcollision="keep.local")
    paths[key] = ent.path
    print(f"{key:12s} {os.path.getsize(ent.path)/1e9:7.2f} GB  {time.time()-t:6.0f}s  {ent.path}")

## 5 · Unzip both archives

In [ ]:
import zipfile
def unzip(zip_path, dest):
    marker = f"{dest}/.done"
    if os.path.exists(marker):
        print("already unzipped:", dest); return
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    open(marker, "w").close()
    print("unzipped:", dest)

unzip(paths["images"], f"{WORK}/images_raw")
unzip(paths["annotations"], f"{WORK}/annotations_raw")

## 6 · Merge annotations into the image folders

Copies (doesn't move) every `<split>/<video>/ann_dir` into the matching image folder.

In [ ]:
from pathlib import Path

def find_split_root(base):
    """Folder that directly contains train/ val/ test/."""
    for p in [Path(base), *sorted(Path(base).rglob("*"))]:
        if p.is_dir() and {"train", "val"} <= {c.name for c in p.iterdir() if c.is_dir()}:
            return p
    raise FileNotFoundError(f"no train/val folders under {base}")

img_root = find_split_root(f"{WORK}/images_raw")
ann_root = find_split_root(f"{WORK}/annotations_raw")
print("images:", img_root, "\nannotations:", ann_root)

copied, missing = 0, []
for ann_dir in sorted(ann_root.glob("*/*/ann_dir")):
    split, video = ann_dir.parent.parent.name, ann_dir.parent.name
    target_video = img_root / split / video
    if not target_video.exists():
        missing.append(f"{split}/{video}")
    dest = target_video / "ann_dir"
    shutil.copytree(ann_dir, dest, dirs_exist_ok=True)
    copied += 1
print(f"merged ann_dir for {copied} videos")
if missing:
    print(f"WARNING: {len(missing)} videos had no image folder:", missing[:10])
DATASET = img_root

## 7 · Check: every annotation has a frame

In [ ]:
import csv, collections
IMG_EXT = {".png", ".jpg", ".jpeg"}
stats = collections.Counter()
unmatched = []
for ann_dir in sorted(DATASET.glob("*/*/ann_dir")):
    video_dir = ann_dir.parent
    images = {p.stem: p for p in video_dir.rglob("*") if p.suffix.lower() in IMG_EXT}
    for j in ann_dir.glob("*.json"):
        stats[video_dir.parent.name + " annotations"] += 1
        # annotation 't80_VID07_000030' -> frame '000030'
        frame = j.stem.split("_")[-1]
        if j.stem in images or frame in images:
            stats[video_dir.parent.name + " matched"] += 1
        else:
            unmatched.append(str(j.relative_to(DATASET)))
for k in sorted(stats): print(f"{k:24s} {stats[k]:>7,}")
print("unmatched annotations:", len(unmatched), unmatched[:5])
print("\nDataset ready at:", DATASET)

## 8 · Save to Google Drive

The Drive account is whichever Google account you approve in the mount pop-up — pick the one that should hold the data.

In [ ]:
if SAVE_ANNOTATIONS_TO_DRIVE or SAVE_IMAGE_ZIP_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    out = Path(DRIVE_DIR) / "data" / "cholecinstanceseg"
    out.mkdir(parents=True, exist_ok=True)
    need_gb = sum(os.path.getsize(paths[k]) for k in (["images"] if SAVE_IMAGE_ZIP_TO_DRIVE else [])) / 1e9
    drive_free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
    print(f"Drive free: {drive_free_gb:.1f} GB, image zip needs: {need_gb:.1f} GB")
    assert drive_free_gb > need_gb + 0.5, "Not enough Google Drive space for the 22.3 GB zip (free plan is 15 GB). Free up space / upgrade, or set SAVE_IMAGE_ZIP_TO_DRIVE = False."
    keys = (["annotations", "metadata"] if SAVE_ANNOTATIONS_TO_DRIVE else []) + (["images"] if SAVE_IMAGE_ZIP_TO_DRIVE else [])
    for key in keys:
        src = Path(paths[key])
        if not (out / src.name).exists():
            print("copying", src.name, "...")
            shutil.copy2(src, out / src.name)
    print("Saved to", out)